# Exercise 2: Code Generation with ReACT Prompting

**Tools Used:** Google Colab, Python, Gemini API, pandas, matplotlib

**Goal:** Use a ReACT-style process to generate Python code, execute it, observe the results, and refine the code based on the observation. The process follows **Thought/Plan → Action → Observation → Refinement**.


In [1]:
# ============================================================
# EXERCISE 2: CODE GENERATION WITH ReACT PROMPTING
# Tools: Google Colab, Python, Gemini API, pandas, matplotlib
# Goal: Use a ReACT-style cycle:
# Thought/Plan -> Action -> Observation -> Refinement
# ============================================================


# --- SETUP ---

!pip install -q -U google-genai

import io
import sys
import time
import pandas as pd
import matplotlib.pyplot as plt

from google import genai
from google.genai import errors
from google.colab import userdata


# Retrieve Gemini API key from Google Colab Secrets
try:
    api_key = userdata.get("GEMINI_API_KEY")

    if not api_key:
        raise ValueError("GEMINI_API_KEY was not found.")

    client = genai.Client(api_key=api_key)

    PRIMARY_MODEL = "gemini-2.5-flash"

    print("Gemini API successfully configured.")
    print(f"Using Gemini model: {PRIMARY_MODEL}")

except Exception as e:
    print(f"Error configuring Gemini API: {e}")
    print("Make sure your Colab Secret is named GEMINI_API_KEY.")
    client = None


# ============================================================
# SAFE GEMINI API CALL
# ============================================================

def safe_generate_content(prompt, system_instruction=None):

    if client is None:
        return None

    config = None

    if system_instruction:
        config = genai.types.GenerateContentConfig(
            system_instruction=system_instruction
        )

    for attempt in range(5):

        try:
            return client.models.generate_content(
                model=PRIMARY_MODEL,
                contents=prompt,
                config=config
            )

        except errors.ServerError:
            if attempt < 4:
                time.sleep(2 ** attempt)

        except errors.APIError as e:
            if e.code == 429:
                if attempt < 4:
                    time.sleep(10 + (attempt * 5))
            else:
                raise RuntimeError(
                    f"API Error ({e.code}): {e.message}"
                )

        except Exception as e:
            if attempt < 4:
                time.sleep(2 ** attempt)
            else:
                raise RuntimeError(
                    f"Unexpected error: {e}"
                )

    raise RuntimeError(
        f"Failed to generate content with {PRIMARY_MODEL} after retries."
    )


# ============================================================
# SAMPLE CUSTOMER SALES DATA
# ============================================================

sample_csv_data = """CustomerID,CustomerName,Region,SalesAmount,Date
C201,Northstar Tech,West,9200.50,2026-01-10
C202,Bluebird LLC,South,7600.00,2026-01-18
C203,Sunrise Market,North,11400.75,2026-02-02
C204,Greenline Co,East,5300.00,2026-02-11
C205,Metro Supply,West,13200.00,2026-02-20
C206,Northstar Tech,West,6800.00,2026-03-04
C207,Bluebird LLC,South,4100.50,2026-03-12
"""

with open("sales_data.csv", "w") as f:
    f.write(sample_csv_data)

print("\nSample sales_data.csv created successfully.")

print("\nSample Data:")
print(pd.read_csv("sales_data.csv"))


# ============================================================
# FUNCTION TO EXECUTE AI-GENERATED PYTHON CODE
# ============================================================

def execute_python_code(code_str):

    old_stdout = sys.stdout
    redirected_output = sys.stdout = io.StringIO()

    error = None

    try:

        exec_globals = {
            "pd": pd,
            "plt": plt
        }

        exec(code_str, exec_globals)

    except Exception as e:

        error = f"{type(e).__name__}: {str(e)}"

    finally:

        sys.stdout = old_stdout

    return redirected_output.getvalue(), error


# ============================================================
# ReACT LOOP
# ============================================================

def run_react_loop(task_description, max_turns=3):

    react_system_instruction = """
You are an expert Python data analyst operating in a ReACT-style
code generation loop.

You will solve the programming task using these stages:

Thought:
Provide a SHORT high-level implementation plan explaining what
the code should do. Do not provide hidden chain-of-thought or
detailed private reasoning.

Action:
Generate executable Python code that follows the plan.

After the code is executed, you will receive an Observation.
Use that Observation to correct or improve the code.

STRICT RESPONSE FORMAT:

Thought:
<Short implementation plan>

Action:
```python
# Executable Python code
```

Rules:
- Return only ONE Thought section and ONE Action block per turn.
- Use only pandas and matplotlib for the data analysis.
- Clearly print important results.
- Include basic error handling.
- Handle missing or invalid data when appropriate.
"""

    current_input = f"""
Task:

{task_description}
"""

    print("\n" + "=" * 70)
    print("STARTING ReACT LOOP")
    print("=" * 70)

    improvement_completed = False

    for turn in range(1, max_turns + 1):

        print(f"\n{'=' * 70}")
        print(f"TURN {turn}")
        print("=" * 70)

        # ----------------------------------------------------
        # THOUGHT + ACTION
        # ----------------------------------------------------

        response = safe_generate_content(
            prompt=current_input,
            system_instruction=react_system_instruction
        )

        if response is None:
            print("No response received from Gemini.")
            break

        response_text = response.text

        print(response_text)


        # ----------------------------------------------------
        # EXTRACT GENERATED PYTHON CODE
        # ----------------------------------------------------

        if "```python" in response_text:

            code_block = (
                response_text
                .split("```python")[1]
                .split("```")[0]
                .strip()
            )

        elif "```" in response_text:

            code_block = (
                response_text
                .split("```")[1]
                .split("```")[0]
                .strip()
            )

        else:

            print("\nNo executable Python code block was found.")
            break


        # ----------------------------------------------------
        # ACTION: EXECUTE GENERATED CODE
        # ----------------------------------------------------

        stdout_value, error_value = execute_python_code(code_block)


        # ----------------------------------------------------
        # OBSERVATION
        # ----------------------------------------------------

        print("\n--- OBSERVATION ---")


        # CASE 1: GENERATED CODE PRODUCED AN ERROR
        if error_value:

            observation = f"""
EXECUTION ERROR:

{error_value}
"""

            print(observation)

            current_input = f"""
Previous response:

{response_text}

Observation:

{observation}

The generated code produced an error.

Analyze the observation and create a corrected version.

Return a new short Thought section and a corrected Action code block.
"""

            time.sleep(5)


        # CASE 2: FIRST VERSION WORKED
        else:

            observation = f"""
SUCCESSFUL EXECUTION

Program Output:

{stdout_value}
"""

            print(observation)


            # ------------------------------------------------
            # FORCE ONE REFINEMENT / IMPROVEMENT CYCLE
            # ------------------------------------------------

            if not improvement_completed:

                print("--- REFINEMENT REQUEST ---")

                improvement_observation = f"""
The initial program executed successfully.

Program Output:

{stdout_value}

However, improve the program so that it is more robust.

The revised code should:

1. Check that the required columns exist.
2. Handle missing SalesAmount values.
3. Handle non-numeric SalesAmount values.
4. Clearly print the total overall sales.
5. Clearly print the customer with the highest total spending.
6. Clearly print sales totals by region.
7. Create and save the regional sales bar chart as
   'sales_by_region.png'.
"""

                print(improvement_observation)

                current_input = f"""
Previous response:

{response_text}

Observation:

{improvement_observation}

The first version worked, but it can be improved.

Use the observation above to refine the program.

Return a new short Thought section and an improved Action code block.
"""

                improvement_completed = True

                time.sleep(5)


            # ------------------------------------------------
            # SECOND SUCCESS = FINISHED
            # ------------------------------------------------

            else:

                print("\nReACT Task completed successfully!")
                break


# ============================================================
# PROGRAMMING TASK
# ============================================================

task = """
Read the file 'sales_data.csv'.

Complete the following tasks:

1. Calculate the total overall sales.

2. Calculate how much each customer spent across all transactions.

3. Identify the customer with the highest total spending.

4. Calculate total SalesAmount for each Region.

5. Create a bar chart showing total SalesAmount by Region.

6. Save the chart as:
   sales_by_region.png

7. Print the results clearly to stdout.

Inputs:
- sales_data.csv

Expected columns:
- CustomerID
- CustomerName
- Region
- SalesAmount
- Date

Allowed libraries:
- pandas
- matplotlib

Include basic error handling for missing columns, missing values,
or invalid numeric sales data.
"""


# ============================================================
# RUN ReACT
# ============================================================

run_react_loop(task)


# ============================================================
# ITERATION NOTE
# ============================================================

print("\n" + "=" * 70)
print("ITERATION / REFINEMENT NOTE")
print("=" * 70)

print("""
The first generated version was tested by executing the AI-generated
Python code inside Google Colab.

The output from that execution became the Observation in the ReACT
cycle.

If an error occurred, the error was passed back to Gemini so it could
generate corrected code.

If the first version worked successfully, Gemini was still asked to
improve the code by adding stronger validation and error handling.

This demonstrates the ReACT-style process:

Thought / Plan
      ↓
Action
      ↓
Observation
      ↓
Refinement
      ↓
Improved Action
""")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 10.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


Gemini API successfully configured.
Using Gemini model: gemini-2.5-flash

Sample sales_data.csv created successfully.

Sample Data:
  CustomerID    CustomerName Region  SalesAmount        Date
0       C201  Northstar Tech   West      9200.50  2026-01-10
1       C202    Bluebird LLC  South      7600.00  2026-01-18
2       C203  Sunrise Market  North     11400.75  2026-02-02
3       C204    Greenline Co   East      5300.00  2026-02-11
4       C205    Metro Supply   West     13200.00  2026-02-20
5       C206  Northstar Tech   West      6800.00  2026-03-04
6       C207    Bluebird LLC  South      4100.50  2026-03-12

STARTING ReACT LOOP

TURN 1
Action:
```python
import pandas as pd
import matplotlib.pyplot as plt

try:
    df = pd.read_csv('sales_data.csv')
except FileNotFoundError:
    print("Error: 'sales_data.csv' not found. Please ensure the file is in the correct directory.")
    exit()
except Exception as e:
    print(f"Error reading CSV file: {e}")
    exit()

# --- Basic Error Hand